# Market Regime Detection with PCA + K-Means

**Intro to ML — Final Project**

## The one-sentence pitch

> We use unsupervised machine learning (PCA + K-means) on cross-asset market data to discover *market regimes* — distinct "kinds of days" the market goes through — and show that these regimes correspond to meaningfully different levels of future risk. What are regimes? "distinct, persistent periods in financial markets characterized by specific behaviors, macroeconomic drivers, and volatility levels"

## What we are *not* doing

We are **not** trying to beat the market. This is very difficult (efficient market, etc). We aren't a top quant firm, anyone who does say they can beat the market at our undergrad level is probably overfitting. 

## What we *are* doing

We are using ML for what unsupervised learning is genuinely good at: **discovering structure in data without being told what to look for.** Concretely:

1. We feed the model 8 cross-asset price series(etfs)  - equities, bonds, gold, commodities, credit, VIX.
2. We engineer rolling features (returns, volatility, drawdowns, correlations).
3. We let PCA + K-means group the trading days into 4 clusters, *without ever telling it about crises, recessions, or news events*.
4. We then check whether those clusters line up with known periods of market stress, and whether they have different future risk profiles **out of sample**.

## Why this matters (impact)

- **For retail investors:** A regime label gives a simple, interpretable signal — "today looks like a stress day" or "today looks calm" — without needing to read financial news or run a Bloomberg terminal.
- **Academically:** It is a clean demonstration of unsupervised learning recovering economically meaningful structure from raw price data.

## 1. Setup

Standard intro-ML stack: pandas, numpy, scikit-learn, matplotlib. Plus `yfinance` for free public data.

```bash
pip install yfinance pandas numpy scikit-learn matplotlib
```

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

RANDOM_STATE = 42

## 2. Data

We use 8 liquid, publicly-traded instruments that together cover most of the macro asset landscape:

| Ticker | What it represents |
|---|---|
| SPY | U.S. large-cap equities (the S&P 500) |
| QQQ | Nasdaq / growth equities |
| IWM | U.S. small-cap equities |
| TLT | Long-duration U.S. Treasuries |
| GLD | Gold |
| DBC | Broad commodities |
| HYG | High-yield corporate credit |
| ^VIX | Implied volatility (the market's "fear gauge") |

We pull daily data from 2010 onward. Using free, public data keeps the project reproducible — any classmate can re-run this notebook.

In [ ]:
tickers = ["SPY", "QQQ", "IWM", "TLT", "GLD", "DBC", "HYG", "^VIX"]

raw = yf.download(
    tickers,
    start="2010-01-01",
    end=None,
    auto_adjust=True,
    progress=False,
)

# yfinance returns a MultiIndex column structure. With auto_adjust=True, "Close" is adjusted.
prices = raw["Close"].copy()
prices = prices.rename(columns={"^VIX": "VIX"})
prices = prices.dropna(how="all").ffill().dropna()

print("Price data shape:", prices.shape)
print("Date range:", prices.index.min().date(), "to", prices.index.max().date())
display(prices.tail())

## 3. Feature Engineering

Each row of our modeling dataset is one trading day. We describe that day using only information **known up to that day** (no look-ahead). The features fall into four buckets:

1. **Rolling returns** (5d, 20d, 60d) — momentum across different horizons
2. **Rolling realized volatility** (20d, 60d) — how turbulent each asset has been
3. **Rolling drawdowns** (20d, 60d for SPY) — how far below recent highs
4. **VIX level and changes** — the market's forward-looking volatility expectation
5. **Cross-asset correlations** (60d, SPY with each other asset) — are assets moving together or apart?

We also compute **future returns and future volatility** for evaluation purposes only — these are *never* given to the clustering model.

In [ ]:
# Daily log returns (handles compounding cleanly)
returns = np.log(prices / prices.shift(1))
# VIX is a level, not a price to compute returns on — handle separately
vix = prices["VIX"]

features = pd.DataFrame(index=prices.index)

asset_tickers = ["SPY", "QQQ", "IWM", "TLT", "GLD", "DBC", "HYG"]

# 1. Rolling returns at 5d, 20d, 60d
for t in asset_tickers:
    for w in [5, 20, 60]:
        features[f"{t}_ret_{w}d"] = returns[t].rolling(w).sum()

# 2. Rolling realized volatility at 20d, 60d (annualized)
for t in asset_tickers:
    for w in [20, 60]:
        features[f"{t}_vol_{w}d"] = returns[t].rolling(w).std() * np.sqrt(252)

# 3. Drawdowns for SPY (the headline equity index)
for w in [20, 60]:
    rolling_max = prices["SPY"].rolling(w).max()
    features[f"SPY_drawdown_{w}d"] = prices["SPY"] / rolling_max - 1

# 4. VIX level and change
features["VIX_level"] = vix
features["VIX_change_5d"] = vix.diff(5)

# 5. Cross-asset 60-day rolling correlations with SPY
for t in ["TLT", "GLD", "DBC", "HYG"]:
    features[f"SPY_{t}_corr_60d"] = returns["SPY"].rolling(60).corr(returns[t])

# 6. Future outcomes — for evaluation ONLY, never fed to the model
features["future_5d_return"] = returns["SPY"].shift(-5).rolling(5).sum().shift(-4)
features["future_20d_return"] = returns["SPY"].shift(-20).rolling(20).sum().shift(-19)
features["future_5d_vol"] = returns["SPY"].shift(-5).rolling(5).std().shift(-4) * np.sqrt(252)
features["future_20d_vol"] = returns["SPY"].shift(-20).rolling(20).std().shift(-19) * np.sqrt(252)

# Drop rows with NaN (the early period before rolling windows fill in)
data = features.dropna().copy()

print("Modeling dataset shape:", data.shape)
print("Date range:", data.index.min().date(), "to", data.index.max().date())
print(f"Total features (excluding future-* evaluation columns): {len([c for c in data.columns if not c.startswith('future_')])}")

## 4. Train / Test Split

We split chronologically. Crucially, we do **not** shuffle — that would leak future information into the training set and inflate our results.

- **Train:** 2010 to end of 2019 (roughly the post-GFC bull market, taper tantrum, 2015 China scare, 2018 Q4 selloff)
- **Test:** 2020 to present (COVID crash, 2021 melt-up, 2022 inflation/bond selloff, 2023 banking scare, 2024+)

The test period contains regimes the model has never seen, so we don't present fake overfit results.

In [ ]:
split_date = "2020-01-01"

# Columns the model is allowed to see (everything except "future_*")
feature_cols = [c for c in data.columns if not c.startswith("future_")]

train = data.loc[data.index < split_date].copy()
test  = data.loc[data.index >= split_date].copy()

X_train = train[feature_cols]
X_test  = test[feature_cols]

print(f"Train: {len(train)} rows  |  {train.index.min().date()} to {train.index.max().date()}")
print(f"Test:  {len(test)} rows  |  {test.index.min().date()} to {test.index.max().date()}")
print(f"Number of features fed to the model: {len(feature_cols)}")

## 5. PCA: Dimensionality Reduction

We have ~50 features, many of which are correlated (e.g., 20-day volatility across different equity ETFs all move together). Feeding correlated features directly into K-means is a known problem — the algorithm essentially double-counts whichever signal is most repeated.

**PCA fixes this** by transforming our features into orthogonal components, ordered by how much variance they explain. We keep enough components to explain 90% of the training variance.

We fit PCA on the **training set only**, then apply that same transformation to the test set.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

pca = PCA(n_components=0.90, random_state=RANDOM_STATE)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca  = pca.transform(X_test_scaled)

explained = pca.explained_variance_ratio_
cum_explained = np.cumsum(explained)

print(f"PCA reduced {X_train.shape[1]} features to {pca.n_components_} components")
print(f"Cumulative variance explained: {cum_explained[-1]:.1%}")

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(cum_explained) + 1), cum_explained, marker="o")
plt.axhline(0.90, linestyle="--", color="gray", label="90% threshold")
plt.xlabel("Number of Principal Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("PCA Explained Variance")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### What does each principal component mean?

PCA produces abstract components. To make the model interpretable, we look at the **loadings** — which original features each component is built from. This is what lets us say "PC1 looks like a stress factor" instead of leaving PCA as a black box.

In [ ]:
loadings = pd.DataFrame(
    pca.components_.T,
    index=feature_cols,
    columns=[f"PC{i+1}" for i in range(pca.n_components_)],
)

for pc in ["PC1", "PC2", "PC3"]:
    print(f"\n=== {pc} ===")
    top_pos = loadings[pc].sort_values(ascending=False).head(5)
    top_neg = loadings[pc].sort_values(ascending=True).head(5)
    print("Top positive loadings:")
    print(top_pos.round(3).to_string())
    print("Top negative loadings:")
    print(top_neg.round(3).to_string())

## 6. Choosing the Number of Clusters

We need to pick `k` — how many regimes the model should find. Two standard intro-ML tools:

1. **Silhouette score** — measures how well-separated the clusters are. Higher is better.
2. **Elbow method (inertia)** — total within-cluster variance. We look for the "elbow" where adding more clusters stops helping much.

We sweep `k` from 2 to 7 on the training data only.

In [ ]:
k_values = range(2, 8)
rows = []

for k in k_values:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=50)
    labels = km.fit_predict(X_train_pca)
    rows.append({
        "k": k,
        "silhouette": silhouette_score(X_train_pca, labels),
        "inertia": km.inertia_,
    })

model_selection = pd.DataFrame(rows)
display(model_selection.round(4))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.plot(model_selection["k"], model_selection["silhouette"], marker="o")
ax1.set(xlabel="k", ylabel="Silhouette Score", title="Silhouette Score by k")
ax1.grid(True, alpha=0.3)
ax2.plot(model_selection["k"], model_selection["inertia"], marker="o")
ax2.set(xlabel="k", ylabel="Inertia", title="Elbow Method")
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Why we chose k = 4

The silhouette and elbow plots give us a quantitative sense of how many clusters fit the data. But for this project we also have an **interpretability argument**: four regimes is the classic finance framing — calm, stress, recovery/risk-off, and a transition/choppy state.

If the silhouette score strongly preferred a much smaller or much larger `k`, we would discuss that. For our data, k = 4 sits at a reasonable point on both curves and gives us clusters we can name.

## 7. Fit K-Means and Assign Regimes

We fit K-means on the **training PCA features only**, then use the trained model to assign regime labels to the test period. The test labels are out-of-sample predictions — the model has never seen those dates.

In [ ]:
best_k = 4
kmeans = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=50)
#bob
train["regime"] = kmeans.fit_predict(X_train_pca)
test["regime"]  = kmeans.predict(X_test_pca)

data_with_regimes = pd.concat([train, test]).sort_index()

print("Training regime counts:")
print(train["regime"].value_counts().sort_index().to_string())
print("\nTest regime counts:")
print(test["regime"].value_counts().sort_index().to_string())

## 8. Naming the Regimes

K-means only gives us cluster numbers (0, 1, 2, 3). We assign human-readable names *after* clustering, using the **training-period averages** — this keeps the naming free of test-period information.

Our rule:
- The regime with the highest average VIX → **"Stress"**
- The regime with the lowest SPY 20-day volatility → **"Calm"**
- The regime with the worst SPY 20-day return → **"Risk-off / drawdown"**
- The remaining one → **"Choppy / transition"**

In [ ]:
summary_cols = [
    "VIX_level", "SPY_ret_20d", "SPY_vol_20d", "SPY_drawdown_60d",
    "TLT_ret_20d", "GLD_ret_20d", "HYG_ret_20d",
]

train_regime_summary = train.groupby("regime")[summary_cols].mean()

def name_regimes(summary: pd.DataFrame) -> dict:
    names = {}
    stress = summary["VIX_level"].idxmax()
    calm = summary["SPY_vol_20d"].idxmin()
    risk_off = summary["SPY_ret_20d"].idxmin()
    for r in summary.index:
        if r == stress and r != risk_off:
            names[r] = "Stress (high-vol)"
        elif r == calm:
            names[r] = "Calm"
        elif r == risk_off:
            names[r] = "Risk-off / drawdown"
        else:
            names[r] = "Choppy / transition"
    # If two roles collapsed to the same cluster, ensure all 4 names exist
    used = set(names.values())
    for r in summary.index:
        if r not in names:
            for fallback in ["Choppy / transition", "Stress (high-vol)", "Risk-off / drawdown", "Calm"]:
                if fallback not in used:
                    names[r] = fallback
                    used.add(fallback)
                    break
    return names

regime_names = name_regimes(train_regime_summary)

print("Training-period regime characteristics:")
display(train_regime_summary.round(4))

print("\nAssigned names:")
for r in sorted(regime_names):
    print(f"  Regime {r}: {regime_names[r]}")

## 9. The Headline Visual — SPY Price Colored by Regime

This is the single most important chart in the project. We plot the entire SPY price history with each day color-coded by the regime our model assigned to it.

**What to look for:**
- Did the model independently flag known crisis periods (2020 COVID crash, 2022 selloff) as "stress" or "risk-off" — *without ever being told about them*?
- Are calm regimes concentrated in known steady bull-market periods?
- Is the test period (2020+) handled sensibly, even though the model was trained only on 2010–2019?

In [ ]:
plot_df = data_with_regimes.copy()
plot_df["SPY_price"] = prices["SPY"].reindex(plot_df.index)
plot_df["regime_name"] = plot_df["regime"].map(regime_names)

color_map = {
    "Calm": "#2ca02c",
    "Choppy / transition": "#1f77b4",
    "Risk-off / drawdown": "#ff7f0e",
    "Stress (high-vol)": "#d62728",
}

plt.figure(figsize=(14, 6))
for regime_id in sorted(plot_df["regime"].unique()):
    subset = plot_df[plot_df["regime"] == regime_id]
    name = regime_names[regime_id]
    plt.scatter(subset.index, subset["SPY_price"],
                s=10, alpha=0.7, label=f"{name}",
                color=color_map.get(name, "gray"))

plt.axvline(pd.Timestamp(split_date), linestyle="--", color="black", alpha=0.5, label="Train/Test split")
plt.title("SPY Price Colored by Discovered Market Regime")
plt.xlabel("Date")
plt.ylabel("SPY Price")
plt.legend(loc="upper left")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Main Result — Out-of-Sample Future Risk by Regime

This is the quantitative version of our claim: do the regimes actually correspond to different *future* risk?

We compute, for the **test period only**, the average future 5-day volatility and the frequency of negative future returns, broken down by regime.

**A successful result looks like:** the stress regime has materially higher future volatility and a higher rate of negative returns than the calm regime, even though the model was only trained on 2010–2019 data.

In [ ]:
def future_stats(df: pd.DataFrame) -> pd.DataFrame:
    out = df.groupby("regime").agg(
        avg_future_5d_return=("future_5d_return", "mean"),
        avg_future_20d_return=("future_20d_return", "mean"),
        avg_future_5d_vol=("future_5d_vol", "mean"),
        avg_future_20d_vol=("future_20d_vol", "mean"),
        pct_negative_next_5d=("future_5d_return", lambda x: np.mean(x < 0)),
        count=("future_5d_return", "count"),
    )
    out["regime_name"] = out.index.map(regime_names)
    return out

train_future = future_stats(train)
test_future = future_stats(test)

print("=== TRAIN PERIOD (for context / interpretation) ===")
display(train_future.round(4))

print("=== TEST PERIOD (out-of-sample — this is our main result) ===")
display(test_future.round(4))

In [ ]:
# Headline bar chart: future vol by regime, out-of-sample
order = test_future.sort_values("avg_future_5d_vol").index
labels = [regime_names[r] for r in order]
vals = test_future.loc[order, "avg_future_5d_vol"].values

plt.figure(figsize=(9, 5))
bars = plt.bar(labels, vals, color=[color_map.get(l, "gray") for l in labels])
plt.title("Out-of-Sample Future 5-Day Volatility by Regime\n(Test Period: 2020 onward)")
plt.ylabel("Annualized Future 5-Day Volatility")
plt.grid(True, alpha=0.3, axis="y")
for bar, val in zip(bars, vals):
    plt.text(bar.get_x() + bar.get_width() / 2, val, f"{val:.1%}",
             ha="center", va="bottom", fontweight="bold")
plt.tight_layout()
plt.show()

# Compute the stress/calm vol ratio for the talking point
stress_vol = test_future.loc[test_future["regime_name"] == "Stress (high-vol)", "avg_future_5d_vol"]
calm_vol = test_future.loc[test_future["regime_name"] == "Calm", "avg_future_5d_vol"]
if len(stress_vol) and len(calm_vol):
    ratio = stress_vol.iloc[0] / calm_vol.iloc[0]
    print(f"\nOut-of-sample ratio of stress-regime future vol to calm-regime future vol: {ratio:.2f}x")

## 11. Sanity Check — Is K-Means Better Than Just Using VIX?

A fair critic would say: "VIX is *already* a volatility signal. Of course high-VIX days have high future vol. What did your fancy ML model add?"

To check, we group test days by **VIX quartile** (a simple, non-ML baseline) and measure the same future-vol spread. If K-means and VIX-quartiles give similar spreads, then the ML model didn't add a numerical edge — but it may still add **interpretability** (a multi-asset story, not just one number).

NEED to potentially clean up this interpretation.

In [ ]:
test_with_vix_quartile = test.copy()
test_with_vix_quartile["VIX_quartile"] = pd.qcut(
    test_with_vix_quartile["VIX_level"], 4, labels=["Q1 low", "Q2", "Q3", "Q4 high"]
)

vix_baseline = test_with_vix_quartile.groupby("VIX_quartile", observed=True).agg(
    avg_future_5d_vol=("future_5d_vol", "mean"),
    pct_negative_next_5d=("future_5d_return", lambda x: np.mean(x < 0)),
    count=("future_5d_return", "count"),
)

print("VIX-quartile baseline (test period):")
display(vix_baseline.round(4))

# Spread metric: max vol bucket minus min vol bucket
km_spread = test_future["avg_future_5d_vol"].max() - test_future["avg_future_5d_vol"].min()
vix_spread = vix_baseline["avg_future_5d_vol"].max() - vix_baseline["avg_future_5d_vol"].min()

print(f"\nFuture-vol spread (high regime - low regime):")
print(f"  PCA + K-Means: {km_spread:.2%}")
print(f"  VIX quartiles: {vix_spread:.2%}")

## 12. Risk Demonstration — What If a Retail Investor Used This Signal?

**Important framing:** this is not a trading strategy. We are *not* claiming we beat the market. What we are doing is showing the model behaves the way our story says it should — when it flags "stress," real risk in the test period was higher.

**The demonstration:** suppose a cautious retail investor used our regime label as a simple risk signal. When the model says "stress," they cut their SPY exposure to 50%; otherwise, they're 100% invested. We then report what happened to their **realized risk**, not their total return.

We expect:
- ✅ Lower realized volatility
- ✅ Lower maximum drawdown
- ❌ Lower total return (this is the *cost* of risk reduction, not a failure)

That cost-benefit tradeoff is the point: risk management isn't free, and pretending it is would be dishonest.

In [ ]:
high_risk_regime = train_future["avg_future_5d_vol"].idxmax()
print(f"'High-risk' regime identified from TRAIN data: regime {high_risk_regime} ({regime_names[high_risk_regime]})\n")

# Build the exposure-adjusted return series for the test period
risk_demo = test.copy()
risk_demo["SPY_daily_return"] = returns["SPY"].reindex(risk_demo.index)
risk_demo["exposure"] = np.where(risk_demo["regime"] == high_risk_regime, 0.50, 1.00)
# Today's regime decides tomorrow's exposure (no look-ahead)
risk_demo["risk_adj_return"] = risk_demo["exposure"].shift(1) * risk_demo["SPY_daily_return"]
risk_demo["buy_hold_return"] = risk_demo["SPY_daily_return"]
risk_demo = risk_demo.dropna(subset=["risk_adj_return", "buy_hold_return"])

def ann_vol(r):
    return r.std() * np.sqrt(252)

def max_dd(r):
    eq = (1 + r).cumprod()
    return (eq / eq.cummax() - 1).min()

def ann_ret(r):
    return (1 + r).prod() ** (252 / len(r)) - 1

risk_table = pd.DataFrame({
    "Buy & Hold SPY": [
        ann_vol(risk_demo["buy_hold_return"]),
        max_dd(risk_demo["buy_hold_return"]),
        ann_ret(risk_demo["buy_hold_return"]),
    ],
    "Regime-Aware (50% in stress regime)": [
        ann_vol(risk_demo["risk_adj_return"]),
        max_dd(risk_demo["risk_adj_return"]),
        ann_ret(risk_demo["risk_adj_return"]),
    ],
}, index=["Annualized Volatility", "Maximum Drawdown", "Annualized Return (for context only)"])

print("Test-period risk profile comparison:")
display(risk_table.round(4))

In [ ]:
# Two side-by-side bar charts: vol and drawdown.
# Deliberately NOT showing a cumulative-return / growth-of-$1 chart,
# because that would invite a 'did you beat the market?' framing we are not making.

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

vol_vals = [ann_vol(risk_demo["buy_hold_return"]), ann_vol(risk_demo["risk_adj_return"])]
dd_vals  = [max_dd(risk_demo["buy_hold_return"]), max_dd(risk_demo["risk_adj_return"])]
labels   = ["Buy & Hold", "Regime-Aware"]

bars1 = ax1.bar(labels, vol_vals, color=["#1f77b4", "#2ca02c"])
ax1.set_title("Realized Annualized Volatility (lower = less risk)")
ax1.set_ylabel("Annualized Vol")
ax1.grid(True, alpha=0.3, axis="y")
for b, v in zip(bars1, vol_vals):
    ax1.text(b.get_x() + b.get_width() / 2, v, f"{v:.1%}", ha="center", va="bottom", fontweight="bold")

bars2 = ax2.bar(labels, [abs(v) for v in dd_vals], color=["#1f77b4", "#2ca02c"])
ax2.set_title("Maximum Drawdown (lower = less painful)")
ax2.set_ylabel("Max Drawdown (absolute value)")
ax2.grid(True, alpha=0.3, axis="y")
for b, v in zip(bars2, dd_vals):
    ax2.text(b.get_x() + b.get_width() / 2, abs(v), f"{v:.1%}", ha="center", va="bottom", fontweight="bold")

plt.tight_layout()
plt.show()

print("\nReading the result:")
print("- Lower vol and lower drawdown is the evidence that the model's 'stress' label is")
print("  actually identifying risky periods. The model does what we said it does.")
print("- The slightly lower total return is the *cost* of de-risking, not a failure of the model.")

## 13. Limitations

A few honest caveats worth mentioning in the report:

1. **K-means assumes spherical clusters of similar size.** Financial regimes may be more irregular. Methods like Gaussian Mixture Models or HMMs could fit better.
2. **Regime names are post-hoc.** The model only outputs numbers (0–3); we apply human labels using training-period statistics.
3. **Daily rows are not independent.** Our rolling features overlap, so standard statistical tests of significance would need to account for autocorrelation.
4. **The test period includes COVID,** which is a once-in-a-generation event. We should be cautious about how representative the test-period results are.
5. **The risk demonstration is not a trading strategy.** A real deployable strategy would need transaction costs, slippage, walk-forward validation, regime stability checks, and risk controls we did not implement.

## 14. Conclusion

We did **not** try to predict returns or beat the S&P 500 — those are problems where intro-ML projects routinely fool themselves.
  


  
What we *did* show is that PCA + K-means, given 8 cross-asset price series and no labels, can recover **economically meaningful market regimes** that:

- Visually line up with known crisis periods on the SPY chart, including ones in the held-out test set the model never saw during training.
- Have **measurably different future volatility profiles** out of sample.
- Behave the way our story claims — when an investor uses the "stress" label to de-risk, realized volatility and drawdowns drop in the test period.

In plain terms: this is a small, honest demonstration that unsupervised ML can help **characterize market risk** even when it cannot predict market returns. That's a useful and defensible thing for an intro ML project to show.

## Presentation outline (10 slides)

1. **The pitch** — One sentence: ML for *understanding* markets, not *trading* them.
2. **Why not just predict returns?** — It's hard, and any intro project that claims to do it is probably wrong.
3. **Data** — 8 cross-asset ETFs from Yahoo Finance.
4. **Features** — Rolling returns, volatility, drawdowns, VIX, correlations.
5. **Method** — Standardize → PCA → K-means with k=4.
6. **Choosing k** — Silhouette + elbow + interpretability argument.
7. **The headline visual** — SPY price colored by regime. *Point to COVID, point to 2022.*
8. **Main quantitative result** — Out-of-sample future vol by regime, with the stress/calm ratio.
9. **Sanity check** — Comparison with VIX-quartile baseline.
10. **Risk demonstration + limitations** — Vol and drawdown drop; total return is the cost; we are not claiming a trading strategy.